In [4]:
%pip install numpy pandas tokenizers transformers

  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached transformers-5.3.0-py3-none-any.whl.metadata (32 kB)
     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     ------------------ ------------------- 20.5/41.5 kB 330.3 kB/s eta 0:00:01
     -------------------------------------  41.0/41.5 kB 487.6 kB/s eta 0:00:01
     -------------------------------------- 41.5/41.5 kB 400.5 kB/s eta 0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   -------------------- ------------------- 1.4/2.7 MB 44.9 MB/s eta 0:00:01
   ---------------------------------------- 2.7/2.7 MB 43.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/10.7 MB ? eta -:--:--
   -------------- ------------------------- 3.9/10.7 MB 122.6 MB/s eta 0:00:01
   ------------------------------ --------- 8.2/10.7 MB 105.6 MB/s eta 0:00:01
   ---------------------------------------  10.7/10.7 MB 93.0 MB/s eta 0:00:01
   ------------


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130

Looking in indexes: https://download.pytorch.org/whl/cu130
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cpu')
if torch.cuda.is_available():
    # CUDA 사용 가능
    device = torch.device('cuda')
elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
    # Apple Silicon (MPS) 사용 가능
    device = torch.device('mps')
print("사용 장치:", device)

import random
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# windows - cuda
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# apple silicon - mps
if torch.mps.is_available():
    torch.mps.manual_seed(SEED)
    torch.use_deterministic_algorithms(True, warn_only=True)

사용 장치: cuda


# 토큰화 & 임베딩

## 토큰화

In [6]:
from pathlib import Path
from urllib.request import Request, urlopen

url = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings.txt"
output_path = Path("ratings.txt")

if output_path.exists():
    print(f"이미 파일이 존재합니다: {output_path.resolve()}")
else:
    request = Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(request, timeout=60) as response, output_path.open("wb") as f:
        f.write(response.read())
    print(f"다운로드 완료: {output_path.resolve()}")

다운로드 완료: C:\Users\SSAFY\git\ai-03\2_1_Token_Emb\ratings.txt


In [7]:
import pandas as pd
import os

file_list = os.listdir()
for file in file_list:
    if "ratings.txt" == file:
        print('학습에 필요한 파일이 존재합니다!', file)
        df = pd.read_table( (os.getcwd() + '/' + file), encoding='utf-8') # 데이터 프레임으로 보기 편하게 바꿔줍시다!
        df = df.dropna(how = 'any') # 널값을 없애줍니다!
        print('리뷰 갯수 :', len(df))
df.head() 

학습에 필요한 파일이 존재합니다! ratings.txt
리뷰 갯수 : 199992


,id,document,label
0,8112052,어릴때보고 지금다시봐도 재밌어요ㅋㅋ,1
1,8132799,"디자인을 배우는 학생으로, 외국디자이너와 그들이 일군 전통을 통해 발전해가는 문화산...",1
2,4655635,폴리스스토리 시리즈는 1부터 뉴까지 버릴께 하나도 없음.. 최고.,1
3,9251303,와.. 연기가 진짜 개쩔구나.. 지루할거라고 생각했는데 몰입해서 봤다.. 그래 이런...,1
4,10067386,안개 자욱한 밤하늘에 떠 있는 초승달 같은 영화.,1


In [8]:
with open((os.getcwd() + '/' + 'naver_review.txt'), 'w', encoding='utf8') as f:
    f.write('\n'.join(df['document'])) 

## 토크나이저 학습

In [15]:
from tokenizers import BertWordPieceTokenizer

# 빈 tokenizer 생성 : vocabulary_size = 0 인 것을 확인하실 수 있습니다.
tokenizer = BertWordPieceTokenizer(
    lowercase=False,
    strip_accents=False,
)
tokenizer

Tokenizer(vocabulary_size=0, model=BertWordPiece, unk_token=[UNK], sep_token=[SEP], cls_token=[CLS], pad_token=[PAD], mask_token=[MASK], clean_text=True, handle_chinese_chars=True, strip_accents=False, lowercase=False, wordpieces_prefix=##)

In [16]:
data_file = 'naver_review.txt'
vocab_size = 30000
min_frequency = 2
initial_alphabet = []
limit_alphabet = 6000
special_tokens = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]
wordpieces_prefix = "##"
show_progress=True

tokenizer.train(
    files=data_file,
    vocab_size=vocab_size,
    min_frequency=min_frequency,
    initial_alphabet=initial_alphabet,
    limit_alphabet=limit_alphabet,
    special_tokens=special_tokens,
    wordpieces_prefix=wordpieces_prefix,
    show_progress=True,
)

vocab = tokenizer.get_vocab()
print("vocab size : ", len(vocab))
print(sorted(vocab, key=lambda x: vocab[x])[:20]) 

vocab size :  30000
['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/']


In [18]:
sorted(vocab, key=lambda x: vocab[x])[10000:10020]

['최고의영화',
 '여자들',
 '일본은',
 '소재와',
 '##스럽지',
 '##만들어',
 '##어야지',
 '##고싶은',
 '한국영화의',
 '전체적인',
 '영환데',
 '오랫만에',
 '장국영',
 '30년',
 '따위',
 '말았',
 '서울',
 '서양',
 '않아서',
 '유럽']

In [30]:
text = "AI 너무 어려워요"

encoded = tokenizer.encode(text)
print('🌱토큰화 결과:', encoded.tokens)
print('🌱정수 인코딩:', encoded.ids)
print('🌈디코딩 (정수로 인코딩된 토큰을 다시 문자열로 변환) :\n\t', tokenizer.decode(encoded.ids))

🌱토큰화 결과: ['A', '##I', '너무', '어려워', '##요']
🌱정수 인코딩: [37, 3365, 5839, 23451, 3408]
🌈디코딩 (정수로 인코딩된 토큰을 다시 문자열로 변환) :
	 AI 너무 어려워요


## 임베딩

In [31]:
embedding_vector = nn.Embedding(vocab_size, 768)  # 512 + 256
embedding_vector.weight.shape 

torch.Size([30000, 768])

In [32]:
token_id = tokenizer.token_to_id("I")
print("token_id:", token_id)
input_id = torch.tensor([token_id], dtype=torch.long)
print("input_id 차원:", input_id.shape)

vector = embedding_vector(input_id)
print("vector 차원:", vector.shape)
print("vector:", vector) 

token_id: 45
input_id 차원: torch.Size([1])
vector 차원: torch.Size([1, 768])
vector: tensor([[ 6.4841e-01, -1.0238e+00, -7.7645e-01,  5.7085e-01, -1.5327e-01,
          7.7379e-01, -5.1051e-01,  1.1337e+00, -6.7349e-02, -9.0853e-01,
          6.8798e-02,  3.8887e-02, -8.0901e-01, -2.3802e-01,  2.1067e-01,
         -2.1456e-01, -1.4842e+00, -1.4351e+00,  2.9461e-01,  2.2708e-01,
         -6.7155e-01,  9.5239e-01,  8.0877e-01,  4.7899e-01,  2.5174e+00,
          4.5432e-01,  1.9498e-01,  9.4856e-01, -7.0154e-01, -7.1777e-01,
         -2.6971e-01, -7.2407e-01,  1.5772e-01,  5.7138e-01,  5.8452e-01,
         -3.4409e-01,  3.2896e+00,  2.4240e-01,  1.8397e+00,  1.0885e-01,
         -7.4473e-02,  8.6946e-01,  6.1239e-01, -1.5340e+00, -2.7230e+00,
         -1.8774e+00,  4.1927e-02, -6.9266e-01, -2.4213e-01,  1.6916e-01,
         -2.2666e-01,  4.8681e-01, -9.5133e-01, -2.1761e-01, -1.4925e+00,
         -7.4105e-01,  8.6661e-01, -1.3713e+00,  7.7098e-01,  5.4892e-01,
         -8.3279e-02, -1.1647e

## RNN Seq2Seq에 활용하기

In [33]:
word_embeddings = nn.Embedding(num_embeddings=vocab_size,
                               embedding_dim=768,
                               device=device)
print("워드 임베딩 차원 :", word_embeddings.weight.shape)

워드 임베딩 차원 : torch.Size([30000, 768])


In [34]:
input_size: int = word_embeddings.weight.size(1) # RNN의 입력값에 들어갈 임베딩 벡터의 차원입니다.
hidden_size: int = 1024  # RNN 내에서 작동하는 은닉 벡터의 차원수입니다.
num_layers: int = 1  # 쌓을 RNN layer의 개수
bidirectional: bool = False  # 단방향 RNN

rnn = nn.RNN(
    input_size=input_size,
    hidden_size=hidden_size,
    num_layers=num_layers,
    bidirectional=bidirectional,
    device=device,
)

# 초기 hidden state 초기화 (num_layers * num_dirs, hidden_size)
hidden_state_shape: int = (num_layers * (2 if bidirectional else 1), hidden_size)
h_0 = torch.zeros(hidden_state_shape, device=device)
print("h_0의 차원 :", h_0.shape) 

h_0의 차원 : torch.Size([1, 1024])


In [36]:
text: str = "나는 학교에 간다."

# 토큰화를 진행합니다.
encoded = tokenizer.encode(text)

# 토큰의 ids만 꺼냅니다. 정수열로 텐서화를 합니다.
input_ids = encoded.ids
input_ids = torch.tensor(input_ids, dtype=torch.long, device=device)

# 변환된 정수열을 벡터화 합니다.
input_embeds = word_embeddings(input_ids)

In [38]:
outputs = rnn(input_embeds, h_0) # hidden_states, h_n
hidden_states = outputs[0] # (sequence_length, hidden_size)
h_n = outputs[1]

# sequence_length (L) : input_token의 길이 (5)
# embedding_dim (H_in): 입력값의 벡터 차원 수 (768)
# hidden_size (H_out) : hidden state 차원 수 (1024)
# num_dirs (D)        : 방향의 개수 (1)
# num_layers          : layer 개수 (1)
print("워드 임베딩 차원 : ", input_embeds.shape)  # (L, H_in)
print("hidden_states 차원 : ", hidden_states.shape)  # (L, D * H_out)
print("h_n 차원 : ", h_n.shape)  # (D * num_layers, H_out))

if torch.equal(hidden_states[-1].unsqueeze(0), h_n):
    print("hidden_states의 마지막과 h_n이 같습니다.") 

워드 임베딩 차원 :  torch.Size([5, 768])
hidden_states 차원 :  torch.Size([5, 1024])
h_n 차원 :  torch.Size([1, 1024])
hidden_states의 마지막과 h_n이 같습니다.


### RNN Encoder

In [39]:
from abc import ABC, abstractmethod

# 인코더 모델은 RNN을 사용합니다. 아래는 추상화 클래스입니다.
class Encoder(nn.Module, ABC):
    def __init__(self: "Encoder") -> None:
        super().__init__()

    @abstractmethod
    def forward(self: "Encoder", input_ids: torch.Tensor) -> torch.Tensor:
        pass


class RNNEncoder(Encoder):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int,
        hidden_size: int,
        num_layers: int,
        bidirectional: bool,
    ) -> None:
        super().__init__()
        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=bidirectional,
        )

    def forward(self, input_ids):
        # 입력 토큰을 워드 임베딩을 통해 임베딩 변환을 합니다.
        input_embeds = self.word_embeddings(input_ids)

        # RNN을 통해 입력 임베딩을 문맥 벡터(context vector)화 합니다.
        hidden_states, h_n = self.rnn(input_embeds)
        return hidden_states, h_n

vocab_size = 30000
embedding_dim = 768
hidden_size = 1024  # RNN의 hidden size
num_layers = 1  # 쌓을 RNN layer의 개수
bidirectional = False  # 단방향 RNN

rnn_encoder = RNNEncoder(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_size=hidden_size,
    num_layers=num_layers,
    bidirectional=bidirectional
).to(device)

outputs = rnn_encoder(input_ids)
hidden_states = outputs[0]
h_n = outputs[1]
print("hidden_states 차원 : ", hidden_states.shape)  # (L, H_out)
print("h_n 차원 : ", h_n.shape)  # (num_layers*D, H_out) = (1, H_out) 

hidden_states 차원 :  torch.Size([5, 1024])
h_n 차원 :  torch.Size([1, 1024])


### RNN Decoder

In [41]:
class Decoder(nn.Module, ABC):
    def __init__(self: "Decoder"):
        super().__init__()

    @abstractmethod
    def forward(self,
                input_ids: torch.Tensor,
                init_hidden_state: torch.Tensor) -> torch.Tensor:
        pass


class RNNDecoder(Decoder):
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int,
        hidden_size: int,
        num_layers: int,
        bidirectional: bool,
        start_token_id: int,
        end_token_id: int,
    ):
        super().__init__()
        self.start_token_id = start_token_id
        self.register_buffer("input_token", torch.tensor([self.start_token_id], dtype=torch.long))
        self.end_token_id = end_token_id

        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=bidirectional,
        )
        self.fully_connected_layer = nn.Linear(hidden_size, vocab_size)
        self.device = self.word_embeddings.weight.device

    def forward(self, init_hidden_state, max_len: int = 10):
        logits = []
        input_token = self.input_token
        output_token_ids = [input_token.item()] # tensor에서 item()을 사용하여 int로 변환합니다.
        h_n = init_hidden_state # h_n은 encoder의 h_0와 동일한 역할을 합니다. Encoder에서 입력된 문장 전체의 은닉 상태를 나타냅니다.

        # 주어진 `max_len`까지 한 스텝씩 다음에 나올 단어를 예측하는 과제를 수행해주세요.
        for _ in range(max_len):
            # 1. RNNEncoder의 마지막 `h_n`을 입력으로 받아 한 step씩 은닉 벡터를 만들어줍니다.
            embedded = self.word_embeddings(input_token)  # 직전 입력 토큰만 사용 (1, H_in)
            _, h_n = self.rnn(embedded, h_n) # 마지막 state만 사용 (1, H_out)

            # 2. 위에서 만든 은닉벡터를 `nn.Linear`를 거쳐 토큰을 예측하여 `logit`을 만들어줍니다.
            logit = self.fully_connected_layer(h_n)  # (1, vocab_size)
            logit = logit.squeeze()
            logits.append(logit)

            # 3. `logit`을 기반으로 예측된 토큰인 `output_token`을 `argmax`를 통해 찾아줍니다.
            output_token = torch.argmax(logit, dim=-1).unsqueeze(0)
            output_token_ids.append(output_token.item())

            # 4. 만약 출력값이 문장의 종료를 나타내는 [SEP]이 나오면 `break`를 통해 생성을 멈추고,
            # 아닐 경우 `input_token`에 `output_token`을 할당하여 루프를 계속 진행시킵니다.
            if output_token == self.end_token_id:
                # 문장의 종료를 의미하는 special token([SEP])이 나왔다면 추론(생성)을 종료합니다.
                break
            else:
                input_token = output_token

        logits = torch.stack(logits, dim=0)  # (max_len, vocab_size)
        return logits, output_token_ids

start_token_id: int = tokenizer.encode("[CLS]").ids[0]
end_token_id: int = tokenizer.encode("[SEP]").ids[0]

vocab_size: int = 30000
embedding_dim: int = 768
hidden_size: int = 1024  # RNN의 hidden size
num_layers: int = 1  # 쌓을 RNN layer의 개수
bidirectional: bool = False  # 단방향 RNN

rnn_decoder = RNNDecoder(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_size=hidden_size,
    num_layers=num_layers,
    bidirectional=bidirectional,
    start_token_id=start_token_id,
    end_token_id=end_token_id,
).to(device)
logits, output_token_ids = rnn_decoder(h_n)
output_texts = tokenizer.decode(output_token_ids)
print(output_texts) 

안고 뱁 그레 that 륙 어리석은 킬링타임용도 N 주인공의 끝난


In [42]:
class RNNSeq2Seq(nn.Module):
    def __init__(self: "RNNSeq2Seq", encoder: nn.Module, decoder: nn.Module) -> None:
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self: "RNNSeq2Seq", input_ids):
        _, context_vector = self.encoder(input_ids) # encoder에서 생성한 context_vector(h_n)을 decoder layer로 전달
        logits, output_tokens = self.decoder(context_vector)
        return logits, output_tokens

seq2seq = RNNSeq2Seq(rnn_encoder, rnn_decoder)
logits, output_tokens = seq2seq(input_ids)
output_token_ids = logits.argmax(dim=-1)
output_texts = tokenizer.decode(output_token_ids.tolist())
print(output_texts) 

안고 뱁 그레 that 륙 어리석은 킬링타임용도 N 주인공의 끝난
